# 03 - Full Ensemble with Subset Training

**Architecture:**
- 12 models across 5 groups (GBDT, Linear, Forest, Neural, AutoML)
- TabPFN/KNN trained by subset (6 models each)
- Regularized ensemble weights
- 15% holdout by UBID for final validation

**Key Improvements:**
1. Subset training for TabPFN (pre/post x drv10/15/30)
2. Entropy + variance regularization on ensemble weights
3. Pre vs Post feature importance analysis
4. Comprehensive holdout report
5. Training time tracking

## Table of Contents
1. [Setup & Dependencies](#1-setup)
2. [Configuration](#2-config)
3. [Data Loading](#3-data)
4. [Preprocessing](#4-preprocess)
5. [Model Training](#5-training)
6. [Subset Models](#6-subset)
7. [Ensemble](#7-ensemble)
8. [Analysis](#8-analysis)
9. [Holdout Validation](#9-holdout)
10. [Save & Submit](#10-save)

---
<a id='1-setup'></a>
## 1. Setup & Dependencies

In [ ]:
# Environment detection
import os
import sys

try:
    import google.colab
    USE_COLAB = True
except ImportError:
    USE_COLAB = False

PROJECT_NAME = '03_Rice_Datathon_Colab'
print(f"Environment: {'Google Colab' if USE_COLAB else 'Local'}")

In [ ]:
# Set up project paths
if USE_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = f'/content/drive/MyDrive/{PROJECT_NAME}'
else:
    PROJECT_ROOT = os.path.dirname(os.getcwd())

sys.path.insert(0, PROJECT_ROOT)
print(f'Project root: {PROJECT_ROOT}')

# Set TabPFN cache BEFORE importing
TABPFN_CACHE = os.path.join(PROJECT_ROOT, 'extra', 'models', 'tabpfn_cache')
os.environ['TABPFN_MODEL_DIR'] = TABPFN_CACHE
os.environ['HF_HOME'] = TABPFN_CACHE
print(f'TabPFN cache: {TABPFN_CACHE}')

In [ ]:
# Install dependencies
def install_if_missing(package, pip_name=None):
    try:
        __import__(package)
    except ImportError:
        import subprocess
        subprocess.check_call(['pip', 'install', '-q', pip_name or package])

install_if_missing('lightgbm')
install_if_missing('xgboost')
install_if_missing('catboost')
install_if_missing('optuna')
# install_if_missing('tabpfn')  # Uncomment if using TabPFN
# install_if_missing('autogluon', 'autogluon')  # Uncomment if using AutoGluon

In [ ]:
# Core imports
import pandas as pd
import numpy as np
import json
import time
import warnings
from pathlib import Path
from datetime import datetime
from scipy.optimize import minimize

# ML imports
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor, Pool
from sklearn.model_selection import KFold, GroupKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import ExtraTreesRegressor, HistGradientBoostingRegressor
from sklearn.linear_model import Ridge, ElasticNet
from sklearn.neighbors import KNeighborsRegressor
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

# TabPFN
try:
    from tabpfn import TabPFNRegressor
    TABPFN_AVAILABLE = True
    print("TabPFN available")
except ImportError:
    TABPFN_AVAILABLE = False
    print("TabPFN not available")

# AutoGluon
try:
    from autogluon.tabular import TabularPredictor
    AUTOGLUON_AVAILABLE = True
    print("AutoGluon available")
except ImportError:
    AUTOGLUON_AVAILABLE = False
    print("AutoGluon not available")

warnings.filterwarnings('ignore')
print("Imports complete")

In [ ]:
# GPU detection (per-library)
GPU_XGB = False
GPU_LGB = False
GPU_CAT = False
GPU_TABPFN = False

try:
    import torch
    if torch.cuda.is_available():
        GPU_TABPFN = True
        # Test XGBoost GPU
        try:
            test_xgb = xgb.XGBRegressor(device='cuda', n_estimators=1)
            test_xgb.fit([[1]], [1])
            GPU_XGB = True
        except: pass
        # Test LightGBM GPU
        try:
            test_lgb = lgb.LGBMRegressor(device='gpu', n_estimators=1, verbose=-1)
            test_lgb.fit([[1]], [1])
            GPU_LGB = True
        except: pass
        # CatBoost usually works
        GPU_CAT = True
except: pass

print(f"GPU Status: XGB={GPU_XGB}, LGB={GPU_LGB}, CAT={GPU_CAT}, TabPFN={GPU_TABPFN}")

---
<a id='2-config'></a>
## 2. Configuration

In [ ]:
# =============================================================================
# BASIC CONFIGURATION
# =============================================================================

# Mode
DRY_RUN = False         # False = full training with more iterations
RESUME = False          # False = retrain models with new features

# Core settings
SEED = 42
N_FOLDS = 5
HOLDOUT_FRACTION = 0.15  # 15% holdout by UBID

# Tuning
N_TRIALS_DRY = 2
N_TRIALS_FULL = 30
N_TRIALS = N_TRIALS_DRY if DRY_RUN else N_TRIALS_FULL

print(f"Mode: {'DRY RUN' if DRY_RUN else 'FULL TRAINING'}")
print(f"Resume: {RESUME}")
print(f"Trials: {N_TRIALS}, Folds: {N_FOLDS}")

In [ ]:
# =============================================================================
# MODEL CONFIGURATION
# =============================================================================

MODELS_TO_TRAIN = [
    # GBDT (train on full data)
    'lgb', 'xgb', 'cat', 'hist',

    # Linear (train on full data)
    'ridge', 'elasticnet',

    # Forest (train on full data)
    'extra_trees',

    # KNN on full data
    'knn',

    # Disabled for now (subset training needs fix)
    # 'tabpfn',      # TabPFN - requires subset fix
    # 'autogluon',   # AutoGluon - poor performance with short time limit
]

# Models to train by subset (for future use when fixed)
SUBSET_MODELS = ['tabpfn', 'knn', 'ridge']

# Subsets: (time_window_tag, trade_area_label)
SUBSETS = [
    ('pre', 'drv10'), ('pre', 'drv15'), ('pre', 'drv30'),
    ('post', 'drv10'), ('post', 'drv15'), ('post', 'drv30'),
]

# Models to load from cache in RESUME mode
RESUME_MODELS = ['lgb', 'xgb', 'cat', 'hist', 'ridge', 'elasticnet', 'extra_trees', 'knn']

print(f"Models to train: {len(MODELS_TO_TRAIN)}")
print(f"Resume models: {RESUME_MODELS}")

In [ ]:
# =============================================================================
# PATHS
# =============================================================================

DIRS = {
    'raw': Path(PROJECT_ROOT) / 'data' / 'raw',
    'processed': Path(PROJECT_ROOT) / 'data' / 'processed',
    'submissions': Path(PROJECT_ROOT) / 'data' / 'submissions',
    'outputs': Path(PROJECT_ROOT) / 'extra' / 'outputs',
    'predictions_oof': Path(PROJECT_ROOT) / 'extra' / 'outputs' / 'predictions_oof',
    'predictions_test': Path(PROJECT_ROOT) / 'extra' / 'outputs' / 'predictions_test',
    'predictions_holdout': Path(PROJECT_ROOT) / 'extra' / 'outputs' / 'predictions_holdout',
    'tuning': Path(PROJECT_ROOT) / 'extra' / 'outputs' / 'tuning',
    'models': Path(PROJECT_ROOT) / 'extra' / 'outputs' / 'models',
}

for d in DIRS.values():
    d.mkdir(parents=True, exist_ok=True)

print("Directories created")

---
<a id='3-data'></a>
## 3. Data Loading

In [ ]:
print("=" * 60)
print("LOADING DATA")
print("=" * 60)

train_df = pd.read_csv(DIRS['processed'] / 'train_clean.csv')
test_df = pd.read_csv(DIRS['processed'] / 'test_clean.csv')

# Load categorical features
with open(DIRS['processed'] / 'categorical_features.json', 'r') as f:
    CAT_FEATURES = json.load(f)

print(f"Train: {train_df.shape}")
print(f"Test: {test_df.shape}")
print(f"Categorical features: {len(CAT_FEATURES)}")

In [ ]:
# =============================================================================
# FEATURE ENGINEERING: Add new features + drop correlated features
# =============================================================================
print("\n--- Feature Engineering ---")

def add_engineered_features(df):
    """Add new composite features."""
    df = df.copy()
    
    # 1. age_cubed - captures non-linear property age effects
    if 'property_age' in df.columns:
        df['age_cubed'] = df['property_age'] ** 3
        print("  Added: age_cubed")
    
    # 2. affordability_index - rent relative to income
    if 'ownrent_avg_rent' in df.columns:
        # Use rent percentile as proxy for affordability if income not available
        if 'rent_percentile' in df.columns:
            df['affordability_index'] = df['ownrent_avg_rent'] / (df['rent_percentile'] + 0.01)
            print("  Added: affordability_index")
    
    # 3. walkable_dining - interaction of walkability and food options
    if 'aarp_score' in df.columns and 'food_total_food' in df.columns:
        df['walkable_dining'] = df['aarp_score'] * df['food_total_food']
        print("  Added: walkable_dining")
    
    # 4. downtown_wide_ring - interaction for urban core properties
    if 'is_downtown' in df.columns and 'msa_ring_num' in df.columns:
        df['downtown_wide_ring'] = df['is_downtown'] * df['msa_ring_num']
        print("  Added: downtown_wide_ring")
    
    return df

# Apply to all datasets
train_df = add_engineered_features(train_df)
test_df = add_engineered_features(test_df)

print(f"\nTrain shape after new features: {train_df.shape}")

# =============================================================================
# DROP HIGHLY CORRELATED FEATURES
# =============================================================================
print("\n--- Dropping Highly Correlated Features ---")

# Features to drop (identified from correlation analysis at threshold=0.8)
CORRELATED_TO_DROP = [
    # Perfect duplicates
    'drivetime_seconds',  # keep drivetime_minutes (1.0 correlation)
    
    # Food count redundancies (all >0.80 correlated)
    'food_count_cheap',      # keep food_count_fast_food (0.997)
    'food_count_moderate',   # keep food_total_food (0.990)
    'food_count_sit_down',   # redundant with food_total_food (0.989)
    'food_count_bar_pub',    # redundant (0.801 with sit_down)
    'food_count_cafe',       # highly correlated with others
    
    # Amenity redundancies
    'total_amenities_ta',    # keep num_food_ta (0.994)
    'num_gas_major_brand_ta',  # keep num_gas_ta (0.809 with grocery)
]

# Only drop columns that exist
cols_to_drop = [c for c in CORRELATED_TO_DROP if c in train_df.columns]

if cols_to_drop:
    train_df = train_df.drop(columns=cols_to_drop)
    test_df = test_df.drop(columns=cols_to_drop)
    print(f"  Dropped {len(cols_to_drop)} correlated features: {cols_to_drop}")
else:
    print("  No correlated features to drop")

print(f"\nFinal train shape: {train_df.shape}")
print(f"Final test shape: {test_df.shape}")

In [ ]:
print("\n--- Creating Holdout Split by UBID ---")

# Get unique UBIDs
train_raw = pd.read_csv(DIRS['raw'] / 'train.csv')
ubids = train_raw['UBID'].unique()
np.random.seed(SEED)
np.random.shuffle(ubids)

n_holdout = int(len(ubids) * HOLDOUT_FRACTION)
holdout_ubids = set(ubids[:n_holdout])

# Create masks
holdout_mask = train_raw['UBID'].isin(holdout_ubids)
train_mask = ~holdout_mask

# Split data
train_data = train_df[train_mask].reset_index(drop=True)
holdout_data = train_df[holdout_mask].reset_index(drop=True)

# Also get UBID for CV
train_ubids = train_raw[train_mask]['UBID'].values

print(f"Train: {len(train_data):,} ({100-HOLDOUT_FRACTION*100:.0f}%)")
print(f"Holdout: {len(holdout_data):,} ({HOLDOUT_FRACTION*100:.0f}%)")
print(f"Holdout UBIDs: {len(holdout_ubids):,}")

In [ ]:
print("\n--- Target Transformation ---")

TARGET = 'target'
TARGET_CLIPPED = 'target_clipped' if 'target_clipped' in train_data.columns else 'target'

y_train_raw = train_data[TARGET].values
y_train = train_data[TARGET_CLIPPED].values
y_holdout_raw = holdout_data[TARGET].values
y_holdout = holdout_data[TARGET_CLIPPED].values if TARGET_CLIPPED in holdout_data.columns else y_holdout_raw

# Log transform if needed
USE_LOG_TRANSFORM = False
if y_train.min() > -0.5:  # Can apply log if mostly positive
    shift = abs(y_train.min()) + 0.1 if y_train.min() < 0 else 0
    y_train_log = np.log1p(y_train + shift)
    y_holdout_log = np.log1p(y_holdout + shift)
    TRANSFORM_PARAMS = {'method': 'log', 'shift': shift}
    USE_LOG_TRANSFORM = True
else:
    y_train_log = y_train
    y_holdout_log = y_holdout
    TRANSFORM_PARAMS = {'method': 'none', 'shift': 0}

print(f"Target: {TARGET_CLIPPED}")
print(f"Transform: {TRANSFORM_PARAMS['method']}")
print(f"Train target stats: mean={y_train.mean():.4f}, std={y_train.std():.4f}")

---
<a id='4-preprocess'></a>
## 4. Preprocessing

In [ ]:
print("=" * 60)
print("PREPROCESSING")
print("=" * 60)

# Identify feature columns
EXCLUDE_COLS = ['target', 'target_clipped', 'target_log', 'id']
FEATURE_COLS = [c for c in train_data.columns if c not in EXCLUDE_COLS]

print(f"Feature columns: {len(FEATURE_COLS)}")

In [ ]:
print("\n--- Encoding Categorical Features ---")

label_encoders = {}
cat_feature_indices = []

for i, col in enumerate(FEATURE_COLS):
    if col in CAT_FEATURES:
        le = LabelEncoder()
        # Fit on combined train + test to handle unseen values
        all_vals = pd.concat([train_data[col], holdout_data[col], test_df[col]]).astype(str)
        le.fit(all_vals)
        train_data[col] = le.transform(train_data[col].astype(str))
        holdout_data[col] = le.transform(holdout_data[col].astype(str))
        test_df[col] = le.transform(test_df[col].astype(str))
        label_encoders[col] = le
        cat_feature_indices.append(i)

print(f"Encoded {len(label_encoders)} categorical features")
print(f"Cat indices: {cat_feature_indices[:5]}...")

In [ ]:
print("\n--- Converting to Numpy Arrays ---")

X_train = train_data[FEATURE_COLS].values.astype(np.float32)
X_holdout = holdout_data[FEATURE_COLS].values.astype(np.float32)
X_test = test_df[FEATURE_COLS].values.astype(np.float32)

# Handle NaN
X_train = np.nan_to_num(X_train, nan=0)
X_holdout = np.nan_to_num(X_holdout, nan=0)
X_test = np.nan_to_num(X_test, nan=0)

print(f"X_train: {X_train.shape}")
print(f"X_holdout: {X_holdout.shape}")
print(f"X_test: {X_test.shape}")

In [ ]:
print("\n--- Creating Scaler for Linear/KNN Models ---")

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_holdout_scaled = scaler.transform(X_holdout)
X_test_scaled = scaler.transform(X_test)

print("Scaler fitted")

In [ ]:
print("\n--- Setting up Cross-Validation ---")

# Group K-Fold by UBID
gkf = GroupKFold(n_splits=N_FOLDS)
cv_splits = list(gkf.split(X_train, y_train, groups=train_ubids))

print(f"CV folds: {N_FOLDS}")
for i, (train_idx, val_idx) in enumerate(cv_splits):
    print(f"  Fold {i+1}: train={len(train_idx):,}, val={len(val_idx):,}")

---
<a id='5-training'></a>
## 5. Model Training

In [ ]:
# =============================================================================
# UTILITY FUNCTIONS
# =============================================================================

def rmse(y_true, y_pred):
    """Calculate RMSE."""
    return np.sqrt(np.mean((y_true - y_pred) ** 2))

def inverse_transform(y, params):
    """Inverse transform predictions."""
    if params['method'] == 'log':
        return np.expm1(y) - params['shift']
    return y

def calculate_gap(train_rmse, val_rmse):
    """Calculate overfitting gap percentage."""
    return (val_rmse - train_rmse) / train_rmse * 100 if train_rmse > 0 else 0

class Timer:
    """Simple timer for tracking training time."""
    def __init__(self):
        self.times = {}
        self.start_time = time.time()
    
    def start(self, name):
        self.times[name] = {'start': time.time()}
    
    def stop(self, name):
        if name in self.times:
            self.times[name]['elapsed'] = time.time() - self.times[name]['start']
            return self.times[name]['elapsed']
        return 0
    
    def summary(self):
        total = time.time() - self.start_time
        print(f"\n{'Model':<20} {'Time (s)':<12} {'% Total':<10}")
        print("-" * 45)
        for name, t in sorted(self.times.items(), key=lambda x: x[1].get('elapsed', 0), reverse=True):
            elapsed = t.get('elapsed', 0)
            pct = elapsed / total * 100 if total > 0 else 0
            print(f"{name:<20} {elapsed:<12.1f} {pct:<10.1f}%")
        print("-" * 45)
        print(f"{'TOTAL':<20} {total:<12.1f}")

timer = Timer()

In [ ]:
def train_lgb(X, y, X_test, X_holdout, cv_splits, params=None):
    """Train LightGBM with CV."""
    if params is None:
        params = {
            'objective': 'regression',
            'metric': 'rmse',
            'boosting_type': 'gbdt',
            'learning_rate': 0.05,
            'num_leaves': 31,
            'max_depth': -1,
            'min_child_samples': 20,
            'subsample': 0.8,
            'colsample_bytree': 0.8,
            'reg_alpha': 0.1,
            'reg_lambda': 0.1,
            'random_state': SEED,
            'verbose': -1,
        }
        if GPU_LGB:
            params['device'] = 'gpu'
    
    n_rounds = 100 if DRY_RUN else 1000
    early_stop = 10 if DRY_RUN else 50
    
    oof = np.zeros(len(X))
    test_preds = np.zeros(len(X_test))
    holdout_preds = np.zeros(len(X_holdout))
    fold_scores = []
    train_scores = []
    feature_importance = np.zeros(X.shape[1])
    
    for fold, (train_idx, val_idx) in enumerate(cv_splits):
        X_tr, X_val = X[train_idx], X[val_idx]
        y_tr, y_val = y[train_idx], y[val_idx]
        
        train_data = lgb.Dataset(X_tr, label=y_tr)
        val_data = lgb.Dataset(X_val, label=y_val)
        
        model = lgb.train(
            params, train_data,
            num_boost_round=n_rounds,
            valid_sets=[val_data],
            callbacks=[
                lgb.early_stopping(early_stop, verbose=False),
                lgb.log_evaluation(0)
            ]
        )
        
        oof[val_idx] = model.predict(X_val)
        test_preds += model.predict(X_test) / len(cv_splits)
        holdout_preds += model.predict(X_holdout) / len(cv_splits)
        
        train_pred = model.predict(X_tr)
        train_scores.append(rmse(y_tr, train_pred))
        fold_scores.append(rmse(y_val, oof[val_idx]))
        feature_importance += model.feature_importance(importance_type='gain') / len(cv_splits)
    
    cv_rmse = rmse(y, oof)
    avg_train = np.mean(train_scores)
    
    return {
        'oof': oof, 'test': test_preds, 'holdout': holdout_preds,
        'cv_rmse': cv_rmse, 'train_rmse': avg_train,
        'gap_pct': calculate_gap(avg_train, cv_rmse),
        'fold_scores': fold_scores,
        'feature_importance': feature_importance,
    }

In [ ]:
def train_xgb(X, y, X_test, X_holdout, cv_splits, params=None):
    """Train XGBoost with CV."""
    if params is None:
        params = {
            'objective': 'reg:squarederror',
            'eval_metric': 'rmse',
            'learning_rate': 0.05,
            'max_depth': 6,
            'min_child_weight': 10,
            'subsample': 0.8,
            'colsample_bytree': 0.8,
            'reg_alpha': 0.1,
            'reg_lambda': 1.0,
            'random_state': SEED,
            'verbosity': 0,
        }
        if GPU_XGB:
            params['device'] = 'cuda'
            params['tree_method'] = 'hist'
    
    n_rounds = 100 if DRY_RUN else 1000
    early_stop = 10 if DRY_RUN else 50
    
    oof = np.zeros(len(X))
    test_preds = np.zeros(len(X_test))
    holdout_preds = np.zeros(len(X_holdout))
    fold_scores = []
    train_scores = []
    feature_importance = np.zeros(X.shape[1])
    
    for fold, (train_idx, val_idx) in enumerate(cv_splits):
        X_tr, X_val = X[train_idx], X[val_idx]
        y_tr, y_val = y[train_idx], y[val_idx]
        
        dtrain = xgb.DMatrix(X_tr, label=y_tr)
        dval = xgb.DMatrix(X_val, label=y_val)
        dtest = xgb.DMatrix(X_test)
        dholdout = xgb.DMatrix(X_holdout)
        
        model = xgb.train(
            params, dtrain,
            num_boost_round=n_rounds,
            evals=[(dval, 'val')],
            early_stopping_rounds=early_stop,
            verbose_eval=False
        )
        
        oof[val_idx] = model.predict(dval)
        test_preds += model.predict(dtest) / len(cv_splits)
        holdout_preds += model.predict(dholdout) / len(cv_splits)
        
        train_pred = model.predict(dtrain)
        train_scores.append(rmse(y_tr, train_pred))
        fold_scores.append(rmse(y_val, oof[val_idx]))
        
        # Feature importance
        imp = model.get_score(importance_type='gain')
        for feat, val in imp.items():
            idx = int(feat.replace('f', ''))
            feature_importance[idx] += val / len(cv_splits)
    
    cv_rmse = rmse(y, oof)
    avg_train = np.mean(train_scores)
    
    return {
        'oof': oof, 'test': test_preds, 'holdout': holdout_preds,
        'cv_rmse': cv_rmse, 'train_rmse': avg_train,
        'gap_pct': calculate_gap(avg_train, cv_rmse),
        'fold_scores': fold_scores,
        'feature_importance': feature_importance,
    }

In [ ]:
def train_cat(X, y, X_test, X_holdout, cv_splits, cat_indices=None, params=None):
    """Train CatBoost with CV."""
    if params is None:
        params = {
            'iterations': 100 if DRY_RUN else 1000,
            'learning_rate': 0.05,
            'depth': 6,
            'l2_leaf_reg': 3,
            'random_seed': SEED,
            'verbose': False,
            'early_stopping_rounds': 10 if DRY_RUN else 50,
        }
        if GPU_CAT:
            params['task_type'] = 'GPU'
    
    oof = np.zeros(len(X))
    test_preds = np.zeros(len(X_test))
    holdout_preds = np.zeros(len(X_holdout))
    fold_scores = []
    train_scores = []
    feature_importance = np.zeros(X.shape[1])
    
    for fold, (train_idx, val_idx) in enumerate(cv_splits):
        X_tr, X_val = X[train_idx], X[val_idx]
        y_tr, y_val = y[train_idx], y[val_idx]
        
        model = CatBoostRegressor(**params)
        model.fit(
            X_tr, y_tr,
            eval_set=(X_val, y_val),
            cat_features=cat_indices,
            verbose=False
        )
        
        oof[val_idx] = model.predict(X_val)
        test_preds += model.predict(X_test) / len(cv_splits)
        holdout_preds += model.predict(X_holdout) / len(cv_splits)
        
        train_pred = model.predict(X_tr)
        train_scores.append(rmse(y_tr, train_pred))
        fold_scores.append(rmse(y_val, oof[val_idx]))
        feature_importance += model.feature_importances_ / len(cv_splits)
    
    cv_rmse = rmse(y, oof)
    avg_train = np.mean(train_scores)
    
    return {
        'oof': oof, 'test': test_preds, 'holdout': holdout_preds,
        'cv_rmse': cv_rmse, 'train_rmse': avg_train,
        'gap_pct': calculate_gap(avg_train, cv_rmse),
        'fold_scores': fold_scores,
        'feature_importance': feature_importance,
    }

In [ ]:
def train_sklearn_model(model_class, X, y, X_test, X_holdout, cv_splits, params=None, 
                        use_scaled=False, X_scaled=None, X_test_scaled=None, X_holdout_scaled=None):
    """Generic sklearn model training. Pass scaled arrays explicitly if use_scaled=True."""
    if params is None:
        params = {}
    
    # Use passed scaled arrays instead of globals
    if use_scaled:
        if X_scaled is None:
            raise ValueError("use_scaled=True but X_scaled not provided")
        X_tr_full = X_scaled
        X_test_use = X_test_scaled
        X_holdout_use = X_holdout_scaled
    else:
        X_tr_full = X
        X_test_use = X_test
        X_holdout_use = X_holdout
    
    oof = np.zeros(len(X))
    test_preds = np.zeros(len(X_test))
    holdout_preds = np.zeros(len(X_holdout))
    fold_scores = []
    train_scores = []
    
    for fold, (train_idx, val_idx) in enumerate(cv_splits):
        X_tr, X_val = X_tr_full[train_idx], X_tr_full[val_idx]
        y_tr, y_val = y[train_idx], y[val_idx]
        
        model = model_class(**params)
        model.fit(X_tr, y_tr)
        
        oof[val_idx] = model.predict(X_val)
        test_preds += model.predict(X_test_use) / len(cv_splits)
        holdout_preds += model.predict(X_holdout_use) / len(cv_splits)
        
        train_pred = model.predict(X_tr)
        train_scores.append(rmse(y_tr, train_pred))
        fold_scores.append(rmse(y_val, oof[val_idx]))
    
    cv_rmse = rmse(y, oof)
    avg_train = np.mean(train_scores)
    
    return {
        'oof': oof, 'test': test_preds, 'holdout': holdout_preds,
        'cv_rmse': cv_rmse, 'train_rmse': avg_train,
        'gap_pct': calculate_gap(avg_train, cv_rmse),
        'fold_scores': fold_scores,
    }

In [ ]:
print("=" * 60)
print("TRAINING ALL MODELS")
print("=" * 60)

results = {}
y = y_train_log if USE_LOG_TRANSFORM else y_train

# =============================================================================
# RESUME MODE: Load cached predictions for models that already trained
# =============================================================================
def load_cached_predictions(model_name):
    """Load cached predictions from .npy files."""
    oof_path = DIRS['predictions_oof'] / f'{model_name}.npy'
    test_path = DIRS['predictions_test'] / f'{model_name}.npy'
    holdout_path = DIRS['predictions_holdout'] / f'{model_name}.npy'

    if oof_path.exists() and test_path.exists() and holdout_path.exists():
        oof = np.load(oof_path)
        test = np.load(test_path)
        holdout = np.load(holdout_path)
        cv_rmse = rmse(y, oof)
        return {
            'oof': oof, 'test': test, 'holdout': holdout,
            'cv_rmse': cv_rmse, 'train_rmse': cv_rmse, 'gap_pct': 0,
            'loaded_from_cache': True
        }
    return None

if RESUME:
    print("\n[RESUME MODE] Loading cached predictions...")
    for model_name in RESUME_MODELS:
        if model_name in MODELS_TO_TRAIN:
            cached = load_cached_predictions(model_name)
            if cached is not None:
                results[model_name] = cached
                print(f"  Loaded {model_name}: CV RMSE = {cached['cv_rmse']:.4f}")
            else:
                print(f"  {model_name}: No cache found, will train")
    print()

# LightGBM
if 'lgb' in MODELS_TO_TRAIN and 'lgb' not in results:
    print("\n[1] LightGBM")
    timer.start('lgb')
    results['lgb'] = train_lgb(X_train, y, X_test, X_holdout, cv_splits)
    timer.stop('lgb')
    print(f"  CV RMSE: {results['lgb']['cv_rmse']:.4f}, Gap: {results['lgb']['gap_pct']:.1f}%")

# XGBoost
if 'xgb' in MODELS_TO_TRAIN and 'xgb' not in results:
    print("\n[2] XGBoost")
    timer.start('xgb')
    results['xgb'] = train_xgb(X_train, y, X_test, X_holdout, cv_splits)
    timer.stop('xgb')
    print(f"  CV RMSE: {results['xgb']['cv_rmse']:.4f}, Gap: {results['xgb']['gap_pct']:.1f}%")

# CatBoost
if 'cat' in MODELS_TO_TRAIN and 'cat' not in results:
    print("\n[3] CatBoost")
    timer.start('cat')
    results['cat'] = train_cat(X_train, y, X_test, X_holdout, cv_splits, None)
    timer.stop('cat')
    print(f"  CV RMSE: {results['cat']['cv_rmse']:.4f}, Gap: {results['cat']['gap_pct']:.1f}%")

# HistGradientBoosting
if 'hist' in MODELS_TO_TRAIN and 'hist' not in results:
    print("\n[4] HistGradientBoosting")
    timer.start('hist')
    params = {'max_iter': 100 if DRY_RUN else 500, 'learning_rate': 0.05, 'random_state': SEED}
    results['hist'] = train_sklearn_model(HistGradientBoostingRegressor, X_train, y, X_test, X_holdout, cv_splits, params)
    timer.stop('hist')
    print(f"  CV RMSE: {results['hist']['cv_rmse']:.4f}, Gap: {results['hist']['gap_pct']:.1f}%")

# Ridge
if 'ridge' in MODELS_TO_TRAIN and 'ridge' not in results:
    print("\n[5] Ridge")
    timer.start('ridge')
    params = {'alpha': 1.0}
    results['ridge'] = train_sklearn_model(Ridge, X_train, y, X_test, X_holdout, cv_splits, params, 
                                           use_scaled=True, X_scaled=X_train_scaled, 
                                           X_test_scaled=X_test_scaled, X_holdout_scaled=X_holdout_scaled)
    timer.stop('ridge')
    print(f"  CV RMSE: {results['ridge']['cv_rmse']:.4f}, Gap: {results['ridge']['gap_pct']:.1f}%")

# ElasticNet
if 'elasticnet' in MODELS_TO_TRAIN and 'elasticnet' not in results:
    print("\n[6] ElasticNet")
    timer.start('elasticnet')
    params = {'alpha': 0.1, 'l1_ratio': 0.5, 'max_iter': 1000}
    results['elasticnet'] = train_sklearn_model(ElasticNet, X_train, y, X_test, X_holdout, cv_splits, params, 
                                                use_scaled=True, X_scaled=X_train_scaled,
                                                X_test_scaled=X_test_scaled, X_holdout_scaled=X_holdout_scaled)
    timer.stop('elasticnet')
    print(f"  CV RMSE: {results['elasticnet']['cv_rmse']:.4f}, Gap: {results['elasticnet']['gap_pct']:.1f}%")

# ExtraTrees
if 'extra_trees' in MODELS_TO_TRAIN and 'extra_trees' not in results:
    print("\n[7] ExtraTrees")
    timer.start('extra_trees')
    params = {'n_estimators': 50 if DRY_RUN else 200, 'max_depth': 15, 'min_samples_leaf': 10, 'random_state': SEED, 'n_jobs': -1}
    results['extra_trees'] = train_sklearn_model(ExtraTreesRegressor, X_train, y, X_test, X_holdout, cv_splits, params)
    timer.stop('extra_trees')
    print(f"  CV RMSE: {results['extra_trees']['cv_rmse']:.4f}, Gap: {results['extra_trees']['gap_pct']:.1f}%")

# KNN
if 'knn' in MODELS_TO_TRAIN and 'knn' not in results:
    print("\n[8] KNN")
    timer.start('knn')
    params = {'n_neighbors': 15, 'weights': 'distance', 'n_jobs': -1}
    results['knn'] = train_sklearn_model(KNeighborsRegressor, X_train, y, X_test, X_holdout, cv_splits, params, 
                                         use_scaled=True, X_scaled=X_train_scaled,
                                         X_test_scaled=X_test_scaled, X_holdout_scaled=X_holdout_scaled)
    timer.stop('knn')
    print(f"  CV RMSE: {results['knn']['cv_rmse']:.4f}, Gap: {results['knn']['gap_pct']:.1f}%")

print(f"\nTotal models loaded/trained: {len(results)}")

---
<a id='6-subset'></a>
## 6. Subset Models (TabPFN/KNN by time_window x trade_area)

In [ ]:
def train_by_subset(model_class, model_name, train_df, test_df, holdout_df,
                    feature_cols, target_col, cv_splits_dict, params=None, use_scaled=False):
    """
    Train model separately for each (time_window_tag, trade_area_label) subset.
    """
    print(f"\n--- {model_name} Subset Training ---")
    
    all_oof = np.zeros(len(train_df))
    all_test = np.zeros(len(test_df))
    all_holdout = np.zeros(len(holdout_df))
    subset_results = {}
    
    for time_tag, drv_label in SUBSETS:
        subset_name = f"{time_tag}_{drv_label}"
        
        # Filter data
        train_mask = (train_df['time_window_tag'] == time_tag) & (train_df['trade_area_label'] == drv_label)
        test_mask = (test_df['time_window_tag'] == time_tag) & (test_df['trade_area_label'] == drv_label)
        holdout_mask = (holdout_df['time_window_tag'] == time_tag) & (holdout_df['trade_area_label'] == drv_label)
        
        train_sub = train_df[train_mask]
        test_sub = test_df[test_mask]
        holdout_sub = holdout_df[holdout_mask]
        
        if len(train_sub) == 0:
            print(f"  {subset_name}: No data, skipping")
            continue
        
        # Prepare data
        X_sub = train_sub[feature_cols].values.astype(np.float32)
        y_sub = train_sub[target_col].values
        X_test_sub = test_sub[feature_cols].values.astype(np.float32) if len(test_sub) > 0 else np.array([])
        X_holdout_sub = holdout_sub[feature_cols].values.astype(np.float32) if len(holdout_sub) > 0 else np.array([])
        
        # Scale if needed
        if use_scaled:
            sub_scaler = StandardScaler()
            X_sub = sub_scaler.fit_transform(X_sub)
            if len(X_test_sub) > 0:
                X_test_sub = sub_scaler.transform(X_test_sub)
            if len(X_holdout_sub) > 0:
                X_holdout_sub = sub_scaler.transform(X_holdout_sub)
        
        # Simple KFold for subset
        kf = KFold(n_splits=min(N_FOLDS, len(train_sub) // 10), shuffle=True, random_state=SEED)
        sub_cv = list(kf.split(X_sub))
        
        # Train
        oof_sub = np.zeros(len(X_sub))
        test_preds_sub = np.zeros(len(X_test_sub)) if len(X_test_sub) > 0 else np.array([])
        holdout_preds_sub = np.zeros(len(X_holdout_sub)) if len(X_holdout_sub) > 0 else np.array([])
        
        for fold, (tr_idx, val_idx) in enumerate(sub_cv):
            model = model_class(**(params or {}))
            model.fit(X_sub[tr_idx], y_sub[tr_idx])
            
            oof_sub[val_idx] = model.predict(X_sub[val_idx])
            if len(test_preds_sub) > 0:
                test_preds_sub += model.predict(X_test_sub) / len(sub_cv)
            if len(holdout_preds_sub) > 0:
                holdout_preds_sub += model.predict(X_holdout_sub) / len(sub_cv)
        
        cv_rmse_sub = rmse(y_sub, oof_sub)
        subset_results[subset_name] = cv_rmse_sub
        print(f"  {subset_name}: n={len(train_sub):,}, CV RMSE={cv_rmse_sub:.4f}")
        
        # Fill back
        train_indices = train_sub.index.values
        for i, idx in enumerate(train_indices):
            loc = train_df.index.get_loc(idx)
            all_oof[loc] = oof_sub[i]
        
        if len(test_sub) > 0:
            test_indices = test_sub.index.values
            for i, idx in enumerate(test_indices):
                loc = test_df.index.get_loc(idx)
                all_test[loc] = test_preds_sub[i]
        
        if len(holdout_sub) > 0:
            holdout_indices = holdout_sub.index.values
            for i, idx in enumerate(holdout_indices):
                loc = holdout_df.index.get_loc(idx)
                all_holdout[loc] = holdout_preds_sub[i]
    
    overall_cv = rmse(train_df[target_col].values, all_oof)
    print(f"  Overall CV RMSE: {overall_cv:.4f}")
    
    return {
        'oof': all_oof, 'test': all_test, 'holdout': all_holdout,
        'cv_rmse': overall_cv, 'subset_results': subset_results,
    }

In [ ]:
# =============================================================================
# FEATURE-SUBSET MODELS FOR DIVERSITY
# =============================================================================
print("=" * 60)
print("TRAINING FEATURE-SUBSET MODELS")
print("=" * 60)

# Define feature groups (updated with new features, removed dropped ones)
FEATURE_GROUPS = {
    'property': [
        'type_main', 'type_sub', 'yearbuilt', 'year_renov', 'numunits', 'areaperunit',
        'property_age', 'vintage_category', 'age_squared', 'age_log', 'age_cubed',  # added age_cubed
        'never_renovated', 'years_since_renov', 'old_never_renovated', 'recently_renovated',
        'is_class_d', 'is_class_a', 'value_tier', 'type_main_te'
    ],
    'geographic': [
        'city', 'state', 'state_metro', 'mrkt_name', 'submrkt_name', 'MARKET',
        'msa_ring', 'msa_norm_dist', 'msa_ring_num', 'zip_metro',
        'is_downtown', 'is_outer_suburb', 'is_sunbelt', 'is_texas',
        'state_te', 'mrkt_name_te', 'msa_ring_te',
        'downtown_wide_ring',  # added downtown_wide_ring
    ],
    'economic': [
        'ownrent_avg_rent', 'ownrent_spread_pct', 'rent_percentile', 'high_rent',
        'supply_baseline_units', 'supply_new_units', 'supply_growth_pct',
        'trade_area_label', 'drv10_area', 'drv15_area', 'drv30_area',
        'is_drv10', 'is_drv30', 'drivetime_minutes',
        'affordability_index',  # added affordability_index
    ],
    'amenities': [
        'num_grocery_ta', 'num_grocery_highend_ta', 'num_gas_ta',
        'num_social_ta', 'num_food_ta', 'num_food_chain_ta',
        'food_total_food', 'food_count_fast_food', 'food_count_high',  # removed dropped food_count_*
        'food_nearest_fast_food_mi', 'food_nearest_sit_down_mi', 'food_nearest_high_price_mi',
        'healthcare_access', 'dining_quality_score', 'has_highend_dining',
        'aarp_score', 'aarp_score_health', 'aarp_score_prox', 'aarp_score_trans',
        'walkable_dining',  # added walkable_dining
    ],
}

# Get indices for each feature group
def get_feature_indices(group_features, all_features):
    """Get indices of features that exist in the dataset."""
    indices = []
    valid_features = []
    for f in group_features:
        if f in all_features:
            indices.append(all_features.index(f))
            valid_features.append(f)
    return indices, valid_features

# Train LightGBM on feature subset
def train_lgb_subset(X_full, y, X_test_full, X_holdout_full, cv_splits, feature_indices, name):
    """Train LightGBM on a subset of features."""
    X = X_full[:, feature_indices]
    X_test = X_test_full[:, feature_indices]
    X_holdout = X_holdout_full[:, feature_indices]
    
    params = {
        'objective': 'regression',
        'metric': 'rmse',
        'boosting_type': 'gbdt',
        'learning_rate': 0.05,
        'num_leaves': 31,
        'max_depth': -1,
        'min_child_samples': 20,
        'subsample': 0.8,
        'colsample_bytree': 0.8,
        'reg_alpha': 0.1,
        'reg_lambda': 0.1,
        'random_state': SEED,
        'verbose': -1,
    }
    
    n_rounds = 100 if DRY_RUN else 1000
    early_stop = 10 if DRY_RUN else 50
    
    oof = np.zeros(len(X))
    test_preds = np.zeros(len(X_test))
    holdout_preds = np.zeros(len(X_holdout))
    
    for fold, (train_idx, val_idx) in enumerate(cv_splits):
        X_tr, X_val = X[train_idx], X[val_idx]
        y_tr, y_val = y[train_idx], y[val_idx]
        
        train_data = lgb.Dataset(X_tr, label=y_tr)
        val_data = lgb.Dataset(X_val, label=y_val)
        
        model = lgb.train(
            params, train_data,
            num_boost_round=n_rounds,
            valid_sets=[val_data],
            callbacks=[
                lgb.early_stopping(early_stop, verbose=False),
                lgb.log_evaluation(0)
            ]
        )
        
        oof[val_idx] = model.predict(X_val)
        test_preds += model.predict(X_test) / len(cv_splits)
        holdout_preds += model.predict(X_holdout) / len(cv_splits)
    
    cv_rmse_val = rmse(y, oof)
    return {
        'oof': oof, 'test': test_preds, 'holdout': holdout_preds,
        'cv_rmse': cv_rmse_val, 'train_rmse': cv_rmse_val, 'gap_pct': 0,
        'n_features': len(feature_indices),
    }

# Train each feature-subset model
y = y_train_log if USE_LOG_TRANSFORM else y_train
subset_models = {}

for group_name, group_features in FEATURE_GROUPS.items():
    model_name = f'lgb_{group_name}'
    indices, valid_features = get_feature_indices(group_features, FEATURE_COLS)
    
    if len(indices) < 5:
        print(f"\n  {model_name}: Only {len(indices)} features, skipping")
        continue
    
    print(f"\n[{group_name.upper()}] Training {model_name} with {len(indices)} features")
    timer.start(model_name)
    
    subset_models[model_name] = train_lgb_subset(
        X_train, y, X_test, X_holdout, cv_splits, indices, model_name
    )
    
    timer.stop(model_name)
    print(f"  CV RMSE: {subset_models[model_name]['cv_rmse']:.4f}")
    
    # Add to main results
    results[model_name] = subset_models[model_name]

# Check correlation with existing models
print("\n" + "=" * 60)
print("CORRELATION ANALYSIS: Subset Models vs Full Models")
print("=" * 60)

base_models = ['lgb', 'xgb', 'cat']
subset_model_names = list(subset_models.keys())

print(f"\n{'Subset Model':<20} {'vs LGB':<12} {'vs XGB':<12} {'vs CAT':<12} {'Avg Corr':<12}")
print("-" * 70)

for sm in subset_model_names:
    corrs = []
    row = f"{sm:<20}"
    for bm in base_models:
        if bm in results and sm in results:
            corr = np.corrcoef(results[sm]['oof'], results[bm]['oof'])[0, 1]
            corrs.append(corr)
            row += f" {corr:<12.4f}"
    avg_corr = np.mean(corrs) if corrs else 0
    row += f" {avg_corr:<12.4f}"
    print(row)

print(f"\nTotal models now: {len(results)}")
print(f"New subset models: {subset_model_names}")

---
<a id='7-ensemble'></a>
## 7. Ensemble

In [ ]:
def simple_average(model_names, results, pred_type='oof'):
    """Simple average of predictions."""
    preds = [results[m][pred_type] for m in model_names if m in results]
    return np.mean(preds, axis=0)

def weighted_average(model_names, results, pred_type='oof'):
    """Weight by inverse CV RMSE."""
    scores = [results[m]['cv_rmse'] for m in model_names if m in results]
    weights = [1/s for s in scores]
    weights = np.array(weights) / sum(weights)
    
    preds = [results[m][pred_type] for m in model_names if m in results]
    return sum(w * p for w, p in zip(weights, preds))

def optimized_weights_regularized(blend_preds, y_true, lambda_entropy=0.1, lambda_variance=0.05):
    """
    Optimize weights with entropy + variance regularization.
    Prevents overfitting to OOF.
    """
    def loss(weights):
        weights = np.abs(weights) / np.sum(np.abs(weights))
        blend = sum(w * p for w, p in zip(weights, blend_preds))
        rmse_loss = np.sqrt(np.mean((y_true - blend) ** 2))
        
        # Entropy regularization (encourage uniform weights)
        entropy = -np.sum(weights * np.log(weights + 1e-8))
        max_entropy = np.log(len(weights))
        entropy_penalty = lambda_entropy * (max_entropy - entropy)
        
        # Variance regularization (penalize extreme weights)
        uniform = 1.0 / len(weights)
        variance_penalty = lambda_variance * np.sum((weights - uniform) ** 2)
        
        return rmse_loss + entropy_penalty + variance_penalty
    
    init = [1/len(blend_preds)] * len(blend_preds)
    result = minimize(loss, init, method='Nelder-Mead', options={'maxiter': 1000})
    weights = np.abs(result.x) / np.sum(np.abs(result.x))
    
    return weights

In [ ]:
print("=" * 60)
print("ENSEMBLE WITH DIVERSE MODELS")
print("=" * 60)

y = y_train_log if USE_LOG_TRANSFORM else y_train

# =============================================================================
# Define model groups
# =============================================================================
FULL_MODELS = ['lgb', 'xgb', 'cat', 'hist', 'extra_trees']
SUBSET_MODELS_LIST = ['lgb_property', 'lgb_geographic', 'lgb_economic', 'lgb_amenities']

full_available = [m for m in FULL_MODELS if m in results]
subset_available = [m for m in SUBSET_MODELS_LIST if m in results]
all_diverse = full_available + subset_available

print(f"Full models: {full_available}")
print(f"Subset models: {subset_available}")
print(f"Total diverse models: {len(all_diverse)}")

# =============================================================================
# STRATEGY 1: Full models only (baseline)
# =============================================================================
print("\n--- Strategy 1: Full Models Only ---")
X_full_train = np.column_stack([results[m]['oof'] for m in full_available])
X_full_test = np.column_stack([results[m]['test'] for m in full_available])
X_full_holdout = np.column_stack([results[m]['holdout'] for m in full_available])

oof_full = np.zeros(len(y))
test_full = np.zeros(len(X_full_test))
holdout_full = np.zeros(len(X_full_holdout))

from sklearn.linear_model import RidgeCV
ridge_full = RidgeCV(alphas=[0.01, 0.1, 1.0, 10.0], cv=5)
ridge_full.fit(X_full_train, y)

for fold, (train_idx, val_idx) in enumerate(cv_splits):
    meta_fold = Ridge(alpha=ridge_full.alpha_)
    meta_fold.fit(X_full_train[train_idx], y[train_idx])
    oof_full[val_idx] = meta_fold.predict(X_full_train[val_idx])
    test_full += meta_fold.predict(X_full_test) / len(cv_splits)
    holdout_full += meta_fold.predict(X_full_holdout) / len(cv_splits)

cv_rmse_full = rmse(y, oof_full)
print(f"  CV RMSE: {cv_rmse_full:.4f} (5 models)")

# =============================================================================
# STRATEGY 2: Full + Subset models (diverse ensemble)
# =============================================================================
print("\n--- Strategy 2: Full + Subset Models (Diverse) ---")
X_diverse_train = np.column_stack([results[m]['oof'] for m in all_diverse])
X_diverse_test = np.column_stack([results[m]['test'] for m in all_diverse])
X_diverse_holdout = np.column_stack([results[m]['holdout'] for m in all_diverse])

print(f"  Stacking {len(all_diverse)} models: {all_diverse}")

oof_diverse = np.zeros(len(y))
test_diverse = np.zeros(len(X_diverse_test))
holdout_diverse = np.zeros(len(X_diverse_holdout))

ridge_diverse = RidgeCV(alphas=[0.01, 0.1, 1.0, 10.0, 100.0], cv=5)
ridge_diverse.fit(X_diverse_train, y)
print(f"  Best alpha: {ridge_diverse.alpha_}")
print(f"  Coefficients: {dict(zip(all_diverse, ridge_diverse.coef_.round(3)))}")

for fold, (train_idx, val_idx) in enumerate(cv_splits):
    meta_fold = Ridge(alpha=ridge_diverse.alpha_)
    meta_fold.fit(X_diverse_train[train_idx], y[train_idx])
    oof_diverse[val_idx] = meta_fold.predict(X_diverse_train[val_idx])
    test_diverse += meta_fold.predict(X_diverse_test) / len(cv_splits)
    holdout_diverse += meta_fold.predict(X_diverse_holdout) / len(cv_splits)

cv_rmse_diverse = rmse(y, oof_diverse)
print(f"  CV RMSE: {cv_rmse_diverse:.4f}")

# =============================================================================
# STRATEGY 3: LightGBM meta-learner on diverse models
# =============================================================================
print("\n--- Strategy 3: LightGBM Meta on Diverse Models ---")

oof_lgb_diverse = np.zeros(len(y))
test_lgb_diverse = np.zeros(len(X_diverse_test))
holdout_lgb_diverse = np.zeros(len(X_diverse_holdout))

lgb_meta_params = {
    'objective': 'regression', 'metric': 'rmse',
    'learning_rate': 0.05, 'num_leaves': 8, 'max_depth': 3,
    'min_child_samples': 50, 'subsample': 0.8, 'colsample_bytree': 0.8,
    'reg_alpha': 1.0, 'reg_lambda': 1.0, 'verbose': -1, 'random_state': SEED,
}

for fold, (train_idx, val_idx) in enumerate(cv_splits):
    dtrain = lgb.Dataset(X_diverse_train[train_idx], label=y[train_idx])
    dval = lgb.Dataset(X_diverse_train[val_idx], label=y[val_idx])
    
    meta_lgb = lgb.train(
        lgb_meta_params, dtrain, num_boost_round=200, valid_sets=[dval],
        callbacks=[lgb.early_stopping(20, verbose=False), lgb.log_evaluation(0)]
    )
    
    oof_lgb_diverse[val_idx] = meta_lgb.predict(X_diverse_train[val_idx])
    test_lgb_diverse += meta_lgb.predict(X_diverse_test) / len(cv_splits)
    holdout_lgb_diverse += meta_lgb.predict(X_diverse_holdout) / len(cv_splits)

cv_rmse_lgb_diverse = rmse(y, oof_lgb_diverse)
print(f"  CV RMSE: {cv_rmse_lgb_diverse:.4f}")

# =============================================================================
# STRATEGY 4: Subset models only (test pure diversity)
# =============================================================================
print("\n--- Strategy 4: Subset Models Only ---")
if len(subset_available) >= 2:
    X_subset_train = np.column_stack([results[m]['oof'] for m in subset_available])
    X_subset_test = np.column_stack([results[m]['test'] for m in subset_available])
    X_subset_holdout = np.column_stack([results[m]['holdout'] for m in subset_available])
    
    oof_subset_only = np.zeros(len(y))
    test_subset_only = np.zeros(len(X_subset_test))
    holdout_subset_only = np.zeros(len(X_subset_holdout))
    
    ridge_subset = RidgeCV(alphas=[0.01, 0.1, 1.0, 10.0], cv=5)
    ridge_subset.fit(X_subset_train, y)
    
    for fold, (train_idx, val_idx) in enumerate(cv_splits):
        meta_fold = Ridge(alpha=ridge_subset.alpha_)
        meta_fold.fit(X_subset_train[train_idx], y[train_idx])
        oof_subset_only[val_idx] = meta_fold.predict(X_subset_train[val_idx])
        test_subset_only += meta_fold.predict(X_subset_test) / len(cv_splits)
        holdout_subset_only += meta_fold.predict(X_subset_holdout) / len(cv_splits)
    
    cv_rmse_subset_only = rmse(y, oof_subset_only)
    print(f"  CV RMSE: {cv_rmse_subset_only:.4f} ({len(subset_available)} models)")
else:
    cv_rmse_subset_only = 999
    oof_subset_only = np.zeros(len(y))
    test_subset_only = np.zeros(len(X_test))
    holdout_subset_only = np.zeros(len(X_holdout))
    print("  Not enough subset models")

# =============================================================================
# STRATEGY 5: Best single model
# =============================================================================
print("\n--- Strategy 5: Best Single Model ---")
best_single = min(results.items(), key=lambda x: x[1]['cv_rmse'])
print(f"  {best_single[0]}: CV RMSE = {best_single[1]['cv_rmse']:.4f}")

# =============================================================================
# Collect all strategies
# =============================================================================
final_ensembles = {
    'full_only': {'oof': oof_full, 'test': test_full, 'holdout': holdout_full, 'cv_rmse': cv_rmse_full},
    'diverse_ridge': {'oof': oof_diverse, 'test': test_diverse, 'holdout': holdout_diverse, 'cv_rmse': cv_rmse_diverse},
    'diverse_lgb': {'oof': oof_lgb_diverse, 'test': test_lgb_diverse, 'holdout': holdout_lgb_diverse, 'cv_rmse': cv_rmse_lgb_diverse},
    'subset_only': {'oof': oof_subset_only, 'test': test_subset_only, 'holdout': holdout_subset_only, 'cv_rmse': cv_rmse_subset_only},
    'best_single': {'oof': results[best_single[0]]['oof'], 'test': results[best_single[0]]['test'], 
                    'holdout': results[best_single[0]]['holdout'], 'cv_rmse': best_single[1]['cv_rmse'], 'name': best_single[0]},
}

print("\n" + "=" * 60)
print("STRATEGY SUMMARY (CV RMSE)")
print("=" * 60)
for name, ens in sorted(final_ensembles.items(), key=lambda x: x[1]['cv_rmse']):
    print(f"  {name:<18}: {ens['cv_rmse']:.4f}")

In [ ]:
# =============================================================================
# ANALYZE DIVERSITY AND SELECT BEST
# =============================================================================
print("\n" + "=" * 60)
print("DIVERSITY ANALYSIS")
print("=" * 60)

# Check correlation between subset models and full models
print("\n--- Correlation: Subset vs Full Models ---")
if subset_available:
    for sm in subset_available:
        corrs_with_full = [np.corrcoef(results[sm]['oof'], results[fm]['oof'])[0, 1] 
                          for fm in full_available]
        avg_corr = np.mean(corrs_with_full)
        min_corr = np.min(corrs_with_full)
        print(f"  {sm}: avg={avg_corr:.4f}, min={min_corr:.4f}")

# Check correlation between subset models themselves
print("\n--- Correlation: Among Subset Models ---")
if len(subset_available) >= 2:
    for i, sm1 in enumerate(subset_available):
        for sm2 in subset_available[i+1:]:
            corr = np.corrcoef(results[sm1]['oof'], results[sm2]['oof'])[0, 1]
            print(f"  {sm1} vs {sm2}: {corr:.4f}")

# Compare old vs new ensemble
print("\n--- Improvement Check ---")
print(f"  Full models only (old):     CV={cv_rmse_full:.4f}")
print(f"  Full + Subset (new):        CV={cv_rmse_diverse:.4f}")
improvement = (cv_rmse_full - cv_rmse_diverse) / cv_rmse_full * 100
print(f"  Improvement: {improvement:.2f}%")

# Select best strategy by CV RMSE
best_strategy = min(final_ensembles, key=lambda x: final_ensembles[x]['cv_rmse'])
print(f"\nBest strategy (by CV): {best_strategy} (CV RMSE: {final_ensembles[best_strategy]['cv_rmse']:.4f})")

---
<a id='8-analysis'></a>
## 8. Analysis

In [ ]:
print("=" * 60)
print("FEATURE IMPORTANCE ANALYSIS")
print("=" * 60)

# Collect importance from models that have it
importance_models = ['lgb', 'xgb', 'cat']
all_importances = {}

for m in importance_models:
    if m in results and 'feature_importance' in results[m]:
        imp = results[m]['feature_importance']
        imp_norm = imp / imp.sum() if imp.sum() > 0 else imp
        all_importances[m] = imp_norm

if all_importances:
    # Average importance
    avg_importance = np.mean(list(all_importances.values()), axis=0)
    
    # Create DataFrame
    imp_df = pd.DataFrame({
        'feature': FEATURE_COLS,
        'avg_importance': avg_importance
    }).sort_values('avg_importance', ascending=False)
    
    print("\n--- Top 15 Features ---")
    for i, row in imp_df.head(15).iterrows():
        print(f"  {row['feature']}: {row['avg_importance']:.4f}")
    
    # Save
    imp_df.to_csv(DIRS['outputs'] / 'feature_importance.csv', index=False)
    print(f"\nSaved to {DIRS['outputs'] / 'feature_importance.csv'}")

In [ ]:
print("\n" + "=" * 60)
print("PRE vs POST COVID ANALYSIS")
print("=" * 60)

# Quick LightGBM to compare importance
for period in ['pre', 'post']:
    mask = train_data['time_window_tag'] == period
    X_period = train_data[mask][FEATURE_COLS].values.astype(np.float32)
    y_period = train_data[mask][TARGET_CLIPPED].values
    
    if len(X_period) > 100:
        quick_model = lgb.LGBMRegressor(n_estimators=100, random_state=SEED, verbose=-1)
        quick_model.fit(X_period, y_period)
        
        imp = quick_model.feature_importances_
        top_idx = np.argsort(imp)[-10:][::-1]
        
        print(f"\n--- {period.upper()}-COVID Top 10 Features ---")
        for i in top_idx:
            print(f"  {FEATURE_COLS[i]}: {imp[i]:.0f}")

---
<a id='9-holdout'></a>
## 9. Holdout Validation

In [ ]:
print("=" * 60)
print("HOLDOUT VALIDATION")
print("=" * 60)

y_holdout_use = y_holdout_log if USE_LOG_TRANSFORM else y_holdout

# Single models
print("\n--- Single Model Performance ---")
print(f"{'Model':<15} {'CV RMSE':<12} {'Holdout RMSE':<14} {'Gap %':<10}")
print("-" * 55)

for model_name, result in sorted(results.items(), key=lambda x: x[1]['cv_rmse']):
    holdout_pred = result['holdout']
    
    # Inverse transform if needed
    if USE_LOG_TRANSFORM:
        holdout_pred_orig = inverse_transform(holdout_pred, TRANSFORM_PARAMS)
        holdout_rmse = rmse(y_holdout_raw, holdout_pred_orig)
        cv_rmse_orig = rmse(y_train_raw, inverse_transform(result['oof'], TRANSFORM_PARAMS))
    else:
        holdout_rmse = rmse(y_holdout, holdout_pred)
        cv_rmse_orig = result['cv_rmse']
    
    gap = (holdout_rmse - cv_rmse_orig) / cv_rmse_orig * 100 if cv_rmse_orig > 0 else 0
    status = "OK" if abs(gap) < 10 else "HIGH"
    print(f"{model_name:<15} {cv_rmse_orig:<12.4f} {holdout_rmse:<14.4f} {gap:<10.1f}%")

# Ensembles
print("\n--- Ensemble Performance ---")
print(f"{'Strategy':<15} {'CV RMSE':<12} {'Holdout RMSE':<14} {'Gap %':<10}")
print("-" * 55)

best_holdout_rmse = float('inf')
best_holdout_strategy = None

for strategy_name, ensemble in final_ensembles.items():
    holdout_pred = ensemble['holdout']
    
    if USE_LOG_TRANSFORM:
        holdout_pred_orig = inverse_transform(holdout_pred, TRANSFORM_PARAMS)
        holdout_rmse = rmse(y_holdout_raw, holdout_pred_orig)
        cv_rmse_orig = rmse(y_train_raw, inverse_transform(ensemble['oof'], TRANSFORM_PARAMS))
    else:
        holdout_rmse = rmse(y_holdout, holdout_pred)
        cv_rmse_orig = ensemble['cv_rmse']
    
    gap = (holdout_rmse - cv_rmse_orig) / cv_rmse_orig * 100 if cv_rmse_orig > 0 else 0
    print(f"{strategy_name:<15} {cv_rmse_orig:<12.4f} {holdout_rmse:<14.4f} {gap:<10.1f}%")
    
    if holdout_rmse < best_holdout_rmse:
        best_holdout_rmse = holdout_rmse
        best_holdout_strategy = strategy_name

print(f"\nBest on holdout: {best_holdout_strategy} (RMSE: {best_holdout_rmse:.4f})")

---
<a id='10-save'></a>
## 10. Save & Submit

In [ ]:
print("=" * 60)
print("SAVING OUTPUTS")
print("=" * 60)

# Save individual model predictions
for model_name, result in results.items():
    np.save(DIRS['predictions_oof'] / f'{model_name}.npy', result['oof'])
    np.save(DIRS['predictions_test'] / f'{model_name}.npy', result['test'])
    np.save(DIRS['predictions_holdout'] / f'{model_name}.npy', result['holdout'])

print(f"Saved predictions for {len(results)} models")

# Save transform params
with open(DIRS['outputs'] / 'transform_params.json', 'w') as f:
    json.dump(TRANSFORM_PARAMS, f, indent=2)

print("Saved transform_params.json")

In [ ]:
print("\n--- Creating Submission ---")

# Use best holdout strategy
final_test_pred = final_ensembles[best_holdout_strategy]['test']

# Inverse transform
if USE_LOG_TRANSFORM:
    final_test_pred = inverse_transform(final_test_pred, TRANSFORM_PARAMS)

# Load test IDs
test_raw = pd.read_csv(DIRS['raw'] / 'test.csv')
submission = pd.DataFrame({
    'id': test_raw['id'],
    'target': final_test_pred
})

# Get CV RMSE for filename
cv_rmse_final = final_ensembles[best_holdout_strategy]['cv_rmse']
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
filename = f'sub_{best_holdout_strategy}_cv{cv_rmse_final:.4f}_{timestamp}.csv'

submission.to_csv(DIRS['submissions'] / filename, index=False)
print(f"Saved: {filename}")
print(f"  Shape: {submission.shape}")
print(f"  Target range: [{submission['target'].min():.4f}, {submission['target'].max():.4f}]")

In [ ]:
print("\n" + "=" * 70)
print("TRAINING COMPLETE")
print("=" * 70)

# Time summary
timer.summary()

# Results summary
print("\n--- Model Summary ---")
print(f"{'Model':<15} {'CV RMSE':<12} {'Gap %':<10} {'Source':<10}")
print("-" * 50)
for m, r in sorted(results.items(), key=lambda x: x[1]['cv_rmse']):
    gap = r.get('gap_pct', 0)
    source = 'cached' if r.get('loaded_from_cache', False) else 'trained'
    print(f"{m:<15} {r['cv_rmse']:<12.4f} {gap:<10.1f}% {source:<10}")

print(f"\n--- Final Results ---")
print(f"Best ensemble strategy: {best_holdout_strategy}")
print(f"CV RMSE: {final_ensembles[best_holdout_strategy]['cv_rmse']:.4f}")
print(f"Holdout RMSE: {best_holdout_rmse:.4f}")
print(f"\nSubmission: {filename}")